# TECO — all plots from the completed run
**CPU → Runtime → Run all**, then allow Drive mounting. Reads your existing `teco_v1_word_erp_compact` compact results. It does not run extraction, change labels, match fixations, reload source EEG or save trial arrays.

Group overviews and regional averages have separate cells, with three neutral comparisons: neutral words from any sentence; neutral words in agent-rated neutral sentences; and neutral words in original dataset-rated neutral sentences. The reader-specific section shows **one combined matrix figure**: 12 reader rows × eight plot columns (whole-head mean, GFP and six regions). Scroll within the matrix to inspect all rows and columns. It also saves as one PNG and one PDF under `analysis/all_plots_v1/reader_matrices/`.

**A full-head signed mean should be almost zero after average reference.** This notebook shows it honestly, and also plots **global field power (GFP)** separately: channel-wise population standard deviation of each reader's class-average waveform, then averaged across readers. GFP is nonnegative, not signed ERP voltage or single-trial GFP. Unequal trial counts and residual noise can influence it. All figures remain descriptive.

New plot files and small CSV reports stay in the existing result folder under `analysis/all_plots_v1/`. Expect a few to a few tens of MB of additional Drive storage, depending on image compression. Running again replaces this gallery's generated files; source averages and extraction checkpoints are read only. No GitHub token is needed. Code stays on GitHub and in temporary Colab storage.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json
from IPython.display import display, Markdown, Image

drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/Thesis/Results/word_sentiment_eeg/teco_v1_word_erp_compact')
REPO = Path('/content/word-sentiment-eeg-teco-plots')
if not (RUN/'manifest.json').is_file():
    raise FileNotFoundError(f'Saved results not found: {RUN}. Check the mounted Google account. No extraction will start.')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary plotting checkout has edits. Preserve those before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','matplotlib'],check=True)
print('Code version:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
# Rebuild outdated plot layouts once from saved averages; reuse only this plotting revision.
PLOTS=RUN/'analysis/all_plots_v1'
existing_gallery=json.loads((PLOTS/'gallery.json').read_text()) if (PLOTS/'gallery.json').is_file() else []
plot_protocol=json.loads((PLOTS/'protocol.json').read_text()) if (PLOTS/'protocol.json').is_file() else {}
reuse_groups=plot_protocol.get('plot_revision')=='regions_without_12_channel_roi_v2' and bool(existing_gallery) and any(x['kind']=='group' for x in existing_gallery) and all((PLOTS/x['path']).is_file() for x in existing_gallery if x['kind']=='group')
command=[sys.executable,'-u',str(REPO/'scripts/plot_teco.py'),'--run-dir',str(RUN)]
if reuse_groups:
    command.append('--reader-matrices-only')
    print('Reusing saved group figures. Creating one combined reader matrix per task.')
run_with_progress(command,RUN/'all_plots_progress.json',activity='TECO: overall, GFP, scalp-region and reader plots')
PLOTS=RUN/'analysis/all_plots_v1'
GALLERY=[x for x in json.loads((PLOTS/'gallery.json').read_text()) if x['kind']!='subject_matrix']
GALLERY+=json.loads((PLOTS/'reader_matrix_gallery.json').read_text())
import importlib, notebook_plot_display
notebook_plot_display=importlib.reload(notebook_plot_display)
show_group_views=notebook_plot_display.show_group_views
show_task_readers=notebook_plot_display.show_task_readers


## Overall averages across readers — all comparisons


In [ ]:
display(Markdown((PLOTS/'REPORT.md').read_text()))
print('Saved gallery:', PLOTS)
show_group_views(GALLERY,PLOTS,['overall'])


## Area-specific averages across readers


In [ ]:
show_group_views(GALLERY,PLOTS,['regions'])


## Electrodes assigned to each scalp region


In [ ]:
display(Image(filename=str(PLOTS/'sensor_regions.png')))


## task1 — individual readers, one combined reader × plot matrix


In [ ]:
show_task_readers(GALLERY,PLOTS,'task1')


In [ ]:
print('All image paths:', PLOTS/'gallery.json')
print('Curve values:', PLOTS/'group_curves.csv.gz')
print('Readers in each comparison:', PLOTS/'comparison_rosters.json')
